# Baseline Model — EV Purchase Prediction (S6E9)

**Competition:** Kaggle Playground Series S6E9 — predict EV purchase probability (ROC-AUC).  
**Public score:** 0.94592

*Analysis was done independently. Code was reviewed and cleaned up with Claude Sonnet 4.6.*

---

## Pipeline

**Nested Target Encoding (NTE)** encodes categorical-like keys using the target variable,  
but without data leakage — each row is encoded using only data from *other* rows.

**Multi-seed bagging** trains the same pipeline 3 times with different random seeds.  
Averaging the predictions reduces variance and stabilizes the final score.

| Step | Detail |
|---|---|
| Outer CV | 5-fold StratifiedKFold × 3 seeds = 15 models |
| TE keys | 6 grouping keys × 2 smoothing strengths = 12 TE features per fold |
| Model | LightGBM (hyperparameters from Optuna) |
| Final prediction | Mean of all 15 test predictions |


## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Fix working directory so relative paths work regardless of where Jupyter is launched
os.chdir(r'C:\\Users\\89671\\Documents\\Hackathons\\kaggle-competitions\\predicting_electric_vehicle_purchases_s6e9')

DATA = Path('data/raw')
PROC = Path('data/processed')

print("Imports OK")
print(f"Working directory: {os.getcwd()}")


## 2. Load Data

In [ ]:
# Processed features (22 columns) built in features.ipynb
X_train = pd.read_csv(PROC / 'X_train.csv')
X_test  = pd.read_csv(PROC / 'X_test.csv')
y_train = pd.read_csv(PROC / 'y_train.csv').squeeze()

# Raw data is needed separately to compute TE keys
# (income, commute, subsidy, anxiety — original string form before encoding)
train_raw = pd.read_csv(DATA / 'train.csv')
test_raw  = pd.read_csv(DATA / 'test.csv')

# Lowercase column names for consistency with features.ipynb
train_raw.columns = train_raw.columns.str.lower()
test_raw.columns  = test_raw.columns.str.lower()

print(f"Train: {X_train.shape},  Test: {X_test.shape}")
print(f"Target positive rate: {y_train.mean():.4f}")


## 3. Nested Target Encoding

### Why naive Target Encoding leaks data

In naive TE, we compute `mean(target)` per group using the entire training set,  
then use that as a feature — meaning each row indirectly "sees" its own target value.  
This is data leakage and leads to overfitting.

### How Nested TE fixes it

For each row in the fold-train:
1. Split fold-train into 5 inner folds
2. Compute TE statistics from inner-train rows only
3. Apply those statistics to the inner-val rows

Each row gets encoded using only *other* rows — no leakage.  
For val/test: encoded using the full fold-train statistics (no leakage,  
since their targets were never seen during model training).

### Bayesian smoothing

```
te = (sum + m × global_mean) / (count + m)
```

- Small group → `te` is pulled toward `global_mean` (regularization)
- Large group → `te` ≈ observed `mean(target)` for that group
- `m=10` — soft smoothing; `m=100` — stronger smoothing for rare groups

**6 keys × 2 smoothing values = 12 TE features added per fold**


In [ ]:
def make_te_keys(df):
    """
    Build 6 grouping keys for target encoding.

    Multiple granularities of income and commute are used:
    - exact values catch fine-grained patterns in synthetic data
    - floor100 / floor1000 create more stable groups with more samples per bucket
    - switch combines the two strongest predictors: subsidy x range anxiety
    """
    income  = df['annual_income_usd'].astype(float)
    commute = df['daily_commute_km'].astype(float)

    return pd.DataFrame({
        'income_exact':     income.round(0).astype(int).astype(str),
        'commute_exact':    commute.round(1).astype(str),
        'income_floor100':  np.floor(income / 100).astype(int).astype(str),
        'income_floor1000': np.floor(income / 1000).astype(int).astype(str),
        'commute_integer':  np.floor(commute).astype(int).astype(str),
        'switch': df['subsidy_available'].astype(str) + '|' + df['range_anxiety_level'].astype(str),
    }, index=df.index)


def nested_target_encode(keys_fold_tr, y_fold_tr, n_inner=5, m_list=(10, 100)):
    """
    Compute leak-free TE for fold-train rows via inner KFold.

    Each row is encoded using statistics from other inner-fold rows only.
    Returns DataFrame with columns: {key}_m10, {key}_m100 for each key.
    """
    global_mean = y_fold_tr.mean()
    result = pd.DataFrame(index=keys_fold_tr.index)

    inner_kf = StratifiedKFold(n_splits=n_inner, shuffle=True, random_state=42)

    for key_col in keys_fold_tr.columns:
        keys = keys_fold_tr[key_col]
        for m in m_list:
            result[f'{key_col}_m{m}'] = 0.0

        for inner_tr_idx, inner_val_idx in inner_kf.split(keys, y_fold_tr):
            inner_tr_keys = keys.iloc[inner_tr_idx]
            inner_tr_y    = y_fold_tr.iloc[inner_tr_idx]

            # Group statistics computed on inner-train rows only
            stats = (
                pd.DataFrame({'k': inner_tr_keys, 'y': inner_tr_y})
                .groupby('k')['y']
                .agg(['sum', 'count'])
            )

            inner_val_keys = keys.iloc[inner_val_idx]
            for m in m_list:
                te_vals = inner_val_keys.map(
                    lambda k: (
                        (stats.loc[k, 'sum'] + m * global_mean) /
                        (stats.loc[k, 'count'] + m)
                        if k in stats.index else global_mean
                    )
                )
                result.loc[keys.index[inner_val_idx], f'{key_col}_m{m}'] = te_vals.values

    return result


def oos_target_encode(keys_oos, keys_fold_tr, y_fold_tr, m_list=(10, 100)):
    """
    Compute TE for out-of-sample rows (val or test) using full fold-train stats.

    No leakage: val/test targets were not used when training the current model.
    """
    global_mean = y_fold_tr.mean()
    result = pd.DataFrame(index=keys_oos.index)

    for key_col in keys_fold_tr.columns:
        stats = (
            pd.DataFrame({'k': keys_fold_tr[key_col], 'y': y_fold_tr})
            .groupby('k')['y']
            .agg(['sum', 'count'])
        )

        for m in m_list:
            te_vals = keys_oos[key_col].map(
                lambda k: (
                    (stats.loc[k, 'sum'] + m * global_mean) /
                    (stats.loc[k, 'count'] + m)
                    if k in stats.index else global_mean
                )
            )
            result[f'{key_col}_m{m}'] = te_vals.values

    return result


print("TE functions ready")
print("6 keys x 2 smoothing values = 12 TE features per fold")


## 4. Model Configuration

LightGBM hyperparameters tuned with Optuna (100 trials, 5-fold CV).  
Shallow trees (`num_leaves=23`) and large `min_child_samples=90` are the main guards against overfitting.


In [ ]:
# Hyperparameters from Optuna search
LGBM_PARAMS = dict(
    n_estimators      = 1069,
    learning_rate     = 0.03359,
    num_leaves        = 23,        # shallow trees — reduces overfitting on synthetic data
    min_child_samples = 90,        # minimum samples per leaf
    subsample         = 0.6799,    # row subsampling per tree
    colsample_bytree  = 0.9624,    # feature subsampling per tree
    reg_alpha         = 2.607e-08,
    reg_lambda        = 8.929e-07,
    objective         = 'binary',
    metric            = 'auc',
    verbosity         = -1,
    n_jobs            = -1,
)

SEEDS   = [42, 0, 123]  # 3 different seeds for bagging
N_FOLDS = 5

print(f"Seeds: {SEEDS},  Folds: {N_FOLDS}")
print(f"Total trainings: {len(SEEDS)} x {N_FOLDS} = {len(SEEDS) * N_FOLDS}")


## 5. Training

`run_seed()` runs a full 5-fold CV with Nested TE for one random seed.  
The three seed results are averaged at the end to form the final prediction.


In [ ]:
def run_seed(seed):
    """
    Run 5-fold CV with Nested Target Encoding for one random seed.

    Returns:
        oof       -- out-of-fold predictions for all train rows
        test_avg  -- mean test predictions across 5 folds
        cv_auc    -- mean validation AUC across 5 folds
    """
    train_keys = make_te_keys(train_raw)
    test_keys  = make_te_keys(test_raw)

    kf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof        = np.zeros(len(X_train))
    test_preds = np.zeros((len(X_test), N_FOLDS))
    fold_aucs  = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
        X_tr,  X_val  = X_train.iloc[tr_idx].copy(), X_train.iloc[val_idx].copy()
        y_tr,  y_val  = y_train.iloc[tr_idx],        y_train.iloc[val_idx]
        keys_tr        = train_keys.iloc[tr_idx]
        keys_val       = train_keys.iloc[val_idx]

        # Nested TE: fold-train rows encoded without their own target values
        te_tr  = nested_target_encode(keys_tr, y_tr)

        # OOS TE: val and test encoded using full fold-train statistics
        te_val  = oos_target_encode(keys_val,  keys_tr, y_tr)
        te_test = oos_target_encode(test_keys, keys_tr, y_tr)

        # Append TE features to static features
        X_tr_full  = pd.concat([X_tr.reset_index(drop=True),   te_tr.reset_index(drop=True)],   axis=1)
        X_val_full = pd.concat([X_val.reset_index(drop=True),  te_val.reset_index(drop=True)],  axis=1)
        X_te_full  = pd.concat([X_test.reset_index(drop=True), te_test.reset_index(drop=True)], axis=1)

        # Train with early stopping on validation AUC
        model = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        model.fit(
            X_tr_full, y_tr,
            eval_set=[(X_val_full, y_val)],
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(period=-1),
            ]
        )

        val_pred              = model.predict_proba(X_val_full)[:, 1]
        oof[val_idx]          = val_pred
        fold_auc              = roc_auc_score(y_val, val_pred)
        fold_aucs.append(fold_auc)
        test_preds[:, fold]   = model.predict_proba(X_te_full)[:, 1]

        print(f"  Fold {fold + 1}: AUC = {fold_auc:.4f}")

    cv_auc   = np.mean(fold_aucs)
    test_avg = test_preds.mean(axis=1)
    return oof, test_avg, cv_auc


print("run_seed() ready")


In [ ]:
all_oof  = []
all_test = []
all_aucs = []

for seed in SEEDS:
    print(f"\n-- Seed {seed} --")
    oof, test_avg, cv_auc = run_seed(seed)
    all_oof.append(oof)
    all_test.append(test_avg)
    all_aucs.append(cv_auc)
    print(f"  CV AUC: {cv_auc:.4f}")

# Average predictions across all seeds
oof_final  = np.mean(all_oof,  axis=0)
test_final = np.mean(all_test, axis=0)

oof_auc = roc_auc_score(y_train, oof_final)

print(f"\n{'='*50}")
print(f"Per-seed CV AUC: {[f'{a:.4f}' for a in all_aucs]}")
print(f"OOF AUC (bagged): {oof_auc:.5f}")


## 6. Save Submission

In [ ]:
sub = pd.read_csv(DATA / 'sample_submission.csv')
sub['Will_Buy_EV'] = test_final   # column name must match sample_submission exactly

out_path = DATA / 'submission_nte_multiseed.csv'
sub.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Rows: {len(sub)},  Columns: {list(sub.columns)}")
sub.head(3)


## 7. Summary

**Final public score: 0.94588** (Kaggle leaderboard, multi-seed submission)

| Experiment | OOF AUC | Public AUC |
|---|---|---|
| Baseline (no TE) | ~0.940 | 0.9418 |
| + Nested Target Encoding | 0.9456 | **0.94592** |
| + Multi-seed bagging (3 seeds) | 0.94586 | 0.94588 |

Bagging provided no meaningful improvement on this dataset —  
the signal is already well-captured by a single seed with Nested TE.

**Best submission:** `submission_nested_te.csv` — single seed, Nested TE only.
